# Project Homework: Applying GitHub Ingestion to OWASP LLM Top 10

This notebook applies the `read_repo_data()` pattern learned in the course to a different repository structure. The OWASP LLM Top 10 repository provides security documentation for large language model applications.

## What's Different

Unlike the course repositories (DataTalks FAQ and Evidently docs):
- **OWASP has NO frontmatter** - files are plain markdown without YAML metadata blocks
- **Different organization** - nested structure with 2_0_vulns/, documentation/, translations/ folders
- **More files** - 542 markdown documents vs 95-1285 in course repos

## Key Learning

The same `read_repo_data()` implementation works across all repositories because:
1. python-frontmatter gracefully handles files without frontmatter (returns empty metadata dict)
2. In-memory zip processing works regardless of repository structure
3. Markdown filtering works uniformly across file organizations

In [1]:
# Import required libraries
import requests
from zipfile import ZipFile
from io import BytesIO
import frontmatter
from pathlib import Path
from typing import List, Dict, Any

## Helper Functions

These functions are copied from course/day1.ipynb (per learning objective - adaptation through reuse, not reinvention). Each function has been enhanced with type hints and docstrings per project/ engineering standards.

In [2]:
def download_repo_zip(repo_owner: str, repo_name: str) -> bytes:
    """Download GitHub repository as zip archive.

    Uses GitHub's codeload API which provides repositories as zip archives
    without requiring authentication for public repos.

    Args:
        repo_owner: GitHub username or organization (e.g., 'OWASP')
        repo_name: Repository name (e.g., 'www-project-top-10-for-large-language-model-applications')

    Returns:
        Binary content of zip archive

    Raises:
        requests.HTTPError: If download fails (repo not found, network error, etc.)
    """
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    print(f"Downloading {repo_owner}/{repo_name} from {url}")

    response = requests.get(url)
    response.raise_for_status()

    print(f"Downloaded {len(response.content)} bytes")
    return response.content

In [3]:
def is_markdown_file(filename: str) -> bool:
    """Check if file is a markdown document (.md or .mdx).

    Args:
        filename: Full path to file in zip archive

    Returns:
        True if file ends with .md or .mdx, False otherwise
    """
    path = Path(filename)
    return path.suffix in ['.md', '.mdx']

In [4]:
def parse_markdown_with_frontmatter(content_bytes: bytes) -> Dict[str, Any]:
    """Parse markdown file extracting frontmatter metadata and content.

    Handles files without frontmatter gracefully - returns empty metadata dict.
    This is critical for OWASP repository which has no frontmatter.

    Args:
        content_bytes: Raw file content as bytes

    Returns:
        Dictionary with:
        - 'metadata': dict of frontmatter key-value pairs (empty {} if no frontmatter)
        - 'content': str of markdown content after frontmatter (or full content if none)
    """
    content_str = content_bytes.decode('utf-8', errors='ignore')
    post = frontmatter.loads(content_str)

    return {
        'metadata': dict(post.metadata),  # Empty dict {} if no frontmatter
        'content': post.content
    }

## Main Function: read_repo_data()

This is the same orchestration logic from the course, proving that good abstractions work across different repository structures.

In [5]:
def read_repo_data(repo_owner: str, repo_name: str) -> List[Dict[str, Any]]:
    """Download GitHub repository and extract markdown documentation with metadata.

    This implementation is identical to course/day1.ipynb because the core
    pattern is universal. The interesting part is how it handles repositories
    with different structures and frontmatter conventions.

    Args:
        repo_owner: GitHub username or organization (e.g., 'OWASP')
        repo_name: Repository name

    Returns:
        List of dictionaries, each containing:
        - 'filename': str - path to file within repository
        - 'metadata': dict - YAML frontmatter key-value pairs (empty {} if no frontmatter)
        - 'content': str - markdown content after frontmatter (or full content if none)
    """
    # Download repository as zip
    zip_content = download_repo_zip(repo_owner, repo_name)

    # Open zip archive in memory
    zip_file = ZipFile(BytesIO(zip_content))

    # Get all files
    all_files = zip_file.namelist()
    print(f"Total files in archive: {len(all_files)}")

    # Filter for markdown files
    markdown_files = [f for f in all_files if is_markdown_file(f)]
    print(f"Markdown files found: {len(markdown_files)}")

    # Process each markdown file
    documents = []
    for filename in markdown_files:
        file_content = zip_file.read(filename)
        parsed = parse_markdown_with_frontmatter(file_content)

        doc = {
            'filename': filename,
            'metadata': parsed['metadata'],
            'content': parsed['content']
        }
        documents.append(doc)

    print(f"Successfully processed {len(documents)} documents")
    return documents

## Testing with OWASP LLM Top 10 Repository

Repository: https://github.com/OWASP/www-project-top-10-for-large-language-model-applications

This repository documents the top 10 security risks for LLM applications. It has a different structure than our course examples and uses plain markdown (no frontmatter).

In [6]:
# Download and process OWASP repository
docs = read_repo_data("OWASP", "www-project-top-10-for-large-language-model-applications")

Downloaded 320788136 bytes
Total files in archive: 1076
Markdown files found: 542
Successfully processed 542 documents


In [7]:
# Display repository characteristics
print(f"OWASP LLM Top 10 Repository Analysis")
print(f"=" * 80)
print(f"Total documents: {len(docs)}")
print(f"Documents with frontmatter: {sum(1 for d in docs if d['metadata'])}")
print(f"Documents without frontmatter: {sum(1 for d in docs if not d['metadata'])}")
print()

# Show sample documents
print(f"Sample documents (first 3):")
for i, doc in enumerate(docs[:3], 1):
    print(f"\n{i}. {doc['filename']}")

    # Handle empty metadata gracefully
    if doc['metadata']:
        print(f"   Metadata: {doc['metadata']}")
    else:
        print(f"   Metadata: None (no frontmatter)")

    # Show content preview
    content_preview = doc['content'][:150].replace('\n', ' ')
    print(f"   Content preview: {content_preview}...")
    print(f"   Total length: {len(doc['content'])} characters")

# Compare to course repos
print(f"\n" + "=" * 80)
print(f"Comparison with Course Repositories:")
print(f"  DataTalks FAQ:     1,285 files, most with frontmatter")
print(f"  Evidently docs:       95 files, most with frontmatter")
print(f"  OWASP LLM Top 10:   {len(docs)} files, NO frontmatter")
print(f"\nKey insight: The same code works across all repos because python-frontmatter")
print(f"gracefully handles files without frontmatter (returns empty metadata dict).")

OWASP LLM Top 10 Repository Analysis
Total documents: 542
Documents with frontmatter: 3
Documents without frontmatter: 539

Sample documents (first 3):

1. www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md
   Metadata: None (no frontmatter)
   Content preview: # [Title of Your PR]  **Key Changes:**  - [ ] List major changes and core updates - [ ] Keep each line under 80 characters - [ ] Focus on the "what" a...
   Total length: 569 characters

2. www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md
   Metadata: None (no frontmatter)
   Content preview: ## Letter from the Project Leads  The OWASP Top 10 for Large Language Model Applications started in 2023 as a community-driven effort to highlight and...
   Total length: 3167 characters

3. www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM01_PromptInjection.md
   Metadata: None (no frontmatter)
   Content preview: ## LLM01:2025 Pro

## What I Learned

**Repository Structure Adaptability:**
- The OWASP repository has 542 markdown files across nested directories (2_0_vulns/, documentation/, resources/)
- Unlike course repos that use frontmatter for metadata, OWASP files are plain markdown
- The `read_repo_data()` implementation handles this gracefully - no code changes needed

**Frontmatter Handling:**
- python-frontmatter library returns empty `metadata: {}` for files without frontmatter
- This makes the implementation robust - works with or without frontmatter
- Important to check `if doc['metadata']:` before displaying metadata fields to avoid confusing output

**Why This Matters for RAG:**
- Some documentation uses frontmatter (Jekyll, Hugo, Next.js sites) for structured metadata
- Other documentation is plain markdown with all structure in headers
- A good RAG ingestion pipeline must handle both cases
- Day 2 chunking will need to adapt strategy based on whether frontmatter provides structure